In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost', 'pyarrow', 'polars'])
import os, gc
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import xgboost as xgb
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Cập nhật đường dẫn theo chuẩn Kaggle
INPUT_DIR = '/kaggle/input/datasets/b22dckh072/feature-engineering/' 
TRAIN_PATH = os.path.join(INPUT_DIR, 'train_interactions.parquet')
META_PATH  = os.path.join(INPUT_DIR, 'filtered_metadata.parquet')
CAND_PATH  = os.path.join(INPUT_DIR, 'candidates_phase2.parquet')
FEAT_OUT   = os.path.join(INPUT_DIR, 'features.parquet')
TEST_PATH  = os.path.join(INPUT_DIR, 'test_interactions.parquet')

# KHAI BÁO CỐ ĐỊNH DANH SÁCH ĐẶC TRƯNG 
# SỬA LẠI DANH SÁCH ĐẶC TRƯNG CHO KHỚP VỚI FILE 05 VÀ TẬN DỤNG HẾT DỮ LIỆU
FEATURES = [
    'user_total_actions', 
    'item_total_sales', 
    'price', 
    'average_rating',
    'rating_number',
    'user_avg_rating_given',
    'sasrec_rank', 
    'lightgcn_rank'
]

In [ ]:
print("Đang nạp file đặc trưng bằng Polars (Siêu tiết kiệm RAM)...")
# Đọc file bằng Polars thay vì Pandas
df_train = pl.read_parquet(FEAT_OUT)

# Tách riêng tập Đặc trưng (X) và Nhãn (y)
X_train = df_train.select(FEATURES)
y_train = df_train.select(['label'])

print("Đang chuyển đổi cấu trúc dữ liệu cho XGBoost...")
# XGBoost hỗ trợ trực tiếp dataframe của Polars, không tốn RAM chuyển đổi
dtrain = xgb.DMatrix(X_train.to_pandas(), label=y_train.to_pandas())

# Giải phóng bảng gốc ngay lập tức vì dtrain đã giữ dữ liệu
del df_train, X_train, y_train
gc.collect()

print("Bắt đầu huấn luyện với GPU...")
params = {
    'tree_method': 'hist', 
    'device': 'cuda',             # Kích hoạt GPU
    'objective': 'binary:logistic', 
    'eval_metric': 'logloss', 
    'max_depth': 6,               
    'scale_pos_weight': 4,        
    'learning_rate': 0.1
}

# Tiến hành huấn luyện
model = xgb.train(params, dtrain, num_boost_round=50) 

# Lưu mô hình
model_path = '/kaggle/working/xgboost_ranking_model.json'
model.save_model(model_path)
print(f"Huấn luyện hoàn tất! Đã lưu mô hình tại: {model_path}")

# Trực quan hóa độ quan trọng
xgb.plot_importance(model, importance_type='gain')
plt.show()

# Dọn dẹp RAM cho các bước sau
del dtrain
gc.collect()